# 4. External validation on Reactome and MeSH (Sections 2.6, 3.7)

1. Build the two sets in the same format as the KEGG test split (CPU is enough; NCBI E-utilities and the Reactome ContentService are queried).
2. Embed them with every model (GPU).
3. Evaluate with the pooling chosen on the KEGG validation split.

No training. Requires notebooks 2 and 3. `BASE/bienc_runs.json` (written by notebook 3) records which run folder is which model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, json, os

REPO = 'https://github.com/GITHUB_ID/pathway-supervised-embedder.git'     # <- this repository
CODE = '/content/pathway-supervised-embedder'
BASE = '/content/drive/MyDrive/kegg-bge'               # Drive folder: data/splits/ and all results
DATA = '/content/data/splits'                          # local copy of the benchmark (faster reads)

if os.path.isdir(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q {REPO} {CODE}
S = f'{CODE}/scripts'
!mkdir -p /content/data {BASE}
!cp -r {BASE}/data/splits /content/data/
def latest(d): return sorted(glob.glob(f'{d}/run-*'))[-1]

EXT = f'{BASE}/external'
EMAIL = 'your@email.com'          # contact address required by NCBI E-utilities
API_KEY = ''                      # optional NCBI API key (faster)
key = f'--api-key {API_KEY}' if API_KEY else ''

## 1. Build the sets
MeSH: 24 Level-4 descriptors, 50 papers each (10–20 min). Reactome: up to 80 pathways × 30 papers (30–60 min). Each folder gets a `meta.json` with the settings and counts. Papers in the KEGG benchmark are excluded.

In [ ]:
!python {S}/build_mesh.py --email {EMAIL} {key} --kegg-dir {DATA} --out-dir {EXT}/mesh

In [ ]:
!python {S}/build_reactome.py --email {EMAIL} {key} --kegg-dir {DATA} --out-dir {EXT}/reactome

## 2. Embeddings (GPU)

In [ ]:
runs = json.load(open(f'{BASE}/bienc_runs.json'))
custom = ' '.join(f'{k}={runs[k]}/best' for k in ['kegg_bge_s42', 'kegg_bge_s1', 'kegg_bge_s2'])
CROSS = sorted(glob.glob(f'{BASE}/result_crossencoder/run-*'))
ours = f'--ours {CROSS[-1]}/best' if CROSS else ''
for db in ['mesh', 'reactome']:
    !python {S}/embed.py --data-dir {EXT}/{db} --train-dir {DATA} --splits test --out-dir {BASE}/emb_{db} --bf16 {ours} --custom {custom} \
        --models longformer biobert scibert pubmedbert specter medcpt bge tfidf random
    !python {S}/embed.py --data-dir {EXT}/{db} --train-dir {DATA} --splits test --out-dir {BASE}/emb_{db} --bf16 --input native --models specter medcpt

Optional, run last (`adapters` may change the transformers version):

In [ ]:
!pip -q install adapters
for db in ['mesh', 'reactome']:
    !python {S}/embed.py --data-dir {EXT}/{db} --train-dir {DATA} --splits test --out-dir {BASE}/emb_{db} --bf16 --models specter2
    !python {S}/embed.py --data-dir {EXT}/{db} --train-dir {DATA} --splits test --out-dir {BASE}/emb_{db} --bf16 --input native --models specter2

## 3. Evaluation
No validation split here, so the pooling chosen on KEGG (`eval/pooling_choice.json`) is reused. MeSH has four levels, Reactome three.

In [ ]:
POOL = f'{BASE}/eval/pooling_choice.json'
!pip -q install tabulate
!python {S}/evaluate.py --data-dir {EXT}/mesh --emb-dir {BASE}/emb_mesh --out-dir {BASE}/eval_mesh \
    --level-keys level1,level2,level3,level4 --pooling-from {POOL} --reference kegg_bge_s42
!python {S}/evaluate.py --data-dir {EXT}/reactome --emb-dir {BASE}/emb_reactome --out-dir {BASE}/eval_reactome \
    --pooling-from {POOL} --reference kegg_bge_s42
from IPython.display import Markdown, display
for db in ['mesh', 'reactome']:
    print(json.dumps(json.load(open(f'{EXT}/{db}/meta.json')), indent=1, ensure_ascii=False)[:1500])
    display(Markdown(open(f'{BASE}/eval_{db}/report_test.md').read()))